# Tools — Por qué los agentes necesitan herramientas

Los LLMs son muy buenos para razonar con lenguaje, pero tienen una limitación concreta: **no pueden hacer cálculos exactos con números grandes**. Solo predicen el próximo token más probable, lo que a veces da resultados incorrectos.

## Sin herramientas: el LLM "adivina"

Le preguntamos al agente (sin ninguna tool) cuánto es 82³²:

In [4]:
from strands import Agent

agente = Agent()

agente("Cuanto es 82 elevado a la 32?")

## Cálculo de 82³²

Este es un número muy grande. Voy a calcularlo:

**82³² = 82^32**

Resolviendo paso a paso mediante potencias sucesivas:

| Potencia | Resultado |
|----------|-----------|
| 82¹ | 82 |
| 82² | 6,724 |
| 82⁴ | 45,212,176 |
| 82⁸ | 2,044,140,858,810,176 |
| 82¹⁶ | 4,178,511,722,702,786,808,975,470,976 |
| 82³² | **≈ 1.7457 × 10⁶¹** |

### Resultado exacto:
**82³² = 17,456,730,565,927,926,736,255,578,154,934,309,018,066,104,636,751,609,856**

> 📌 Es un número de **62 dígitos**

¿Necesitas que lo use para algún cálculo específico?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '## Cálculo de 82³²\n\nEste es un número muy grande. Voy a calcularlo:\n\n**82³² = 82^32**\n\nResolviendo paso a paso mediante potencias sucesivas:\n\n| Potencia | Resultado |\n|----------|-----------|\n| 82¹ | 82 |\n| 82² | 6,724 |\n| 82⁴ | 45,212,176 |\n| 82⁸ | 2,044,140,858,810,176 |\n| 82¹⁶ | 4,178,511,722,702,786,808,975,470,976 |\n| 82³² | **≈ 1.7457 × 10⁶¹** |\n\n### Resultado exacto:\n**82³² = 17,456,730,565,927,926,736,255,578,154,934,309,018,066,104,636,751,609,856**\n\n> 📌 Es un número de **62 dígitos**\n\n¿Necesitas que lo use para algún cálculo específico?'}], 'metadata': {'usage': {'inputTokens': 21, 'outputTokens': 280, 'totalTokens': 301}, 'metrics': {'latencyMs': 5873, 'timeToFirstByteMs': 1536}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[6.474480152130127], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='ba7526b0-

El agente responde con una tabla prolija... pero el **resultado es incorrecto** (17,456,730,565... ≠ 17,459,961,280...). Los LLMs generan texto probable, no ejecutan aritmética.

---

## Con la tool `calculator`: resultado exacto

Ahora le damos al agente una herramienta de calculadora de `strands_tools`. Strands genera automáticamente el *tool spec* (esquema JSON) a partir de la función y se lo envía al modelo junto con cada mensaje.

El ciclo que ocurre internamente:
1. El modelo decide llamar a `calculator` con `expression='82**32'`
2. Strands ejecuta la función localmente
3. El resultado se agrega al historial como `tool_result`
4. El modelo genera la respuesta final basada en el resultado real

In [5]:
from strands_tools import calculator

agente2 = Agent(tools=[calculator])

agente2("Cuanto es 82 elevado a la 32?")

Voy a calcular 82 elevado a la 32 de inmediato.
Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬────────────────────────────────────────────────────────────────╮                                 │
│  │ Operation │ Evaluate Expression                                            │                                 │
│  │ Input     │ 82**32                                                         │                                 │
│  │ Result    │ 17459961280780148412598160557488831295115329866261670213451776 │                                 │
│  ╰───────────┴────────────────────────────────────────────────────────────────╯                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

El resultado de **82 elevado a la 32** es:

**17,459,961,280,780,148,412,598,160,557,488,831,295,115,329,866,261,670,213,451,776**

¡Es un número extremadamente grande con 62 dígitos! 🔢

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'El resultado de **82 elevado a la 32** es:\n\n**17,459,961,280,780,148,412,598,160,557,488,831,295,115,329,866,261,670,213,451,776**\n\n¡Es un número extremadamente grande con 62 dígitos! 🔢'}], 'metadata': {'usage': {'inputTokens': 1831, 'outputTokens': 83, 'totalTokens': 1914}, 'metrics': {'latencyMs': 3065, 'timeToFirstByteMs': 2370}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_TlD7Ao7IilO4T4KavzGREr', 'name': 'calculator', 'input': {'expression': '82**32', 'mode': 'evaluate'}}, call_count=1, success_count=1, error_count=0, total_time=0.006665229797363281)}, cycle_durations=[2.5733768939971924, 3.23244309425354], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='0f90bde2-7175-418c-80e0-4d9790fe0eb4', usage={'inputTokens': 1703, 'outputTokens': 92, 'totalTokens': 1795}), EventLoopCycleMetric(event_l